# TE family x genomic-location tables + TE -> location mapping, for the 10 TE
# families, using GENCODE v47 overlap-based labels (genic-first priority)

In [1]:
1

[1] 1

In [2]:
suppressMessages({ library(data.table); library(rtracklayer); library(GenomicRanges) })

DIR <- "liu_data"
GTF <- "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.annotation.gtf.gz"
fams10 <- c("Retroposon/SVA","SINE/Alu","SINE/MIR","LINE/L1","LINE/L2",
            "LTR","LTR/ERV1","LTR/ERVK","LTR/ERVL","LTR/ERVL-MaLR")
region_levels <- c("Promoter (<=3kb)","5' UTR","3' UTR","Exon","Intron","Downstream","Distal Intergenic")


In [8]:
# GENCODE v47 feature sets in here
gtf <- import(GTF)
tx      <- gtf[gtf$type == "transcript"]
exons   <- reduce(gtf[gtf$type == "exon"], ignore.strand = TRUE)
cds_gr  <- gtf[gtf$type == "CDS"]

In [9]:
# GENCODE carries single 'UTR' type and so need to split 5'/3' by position vs the transcript's CDS span
utr_gr  <- gtf[gtf$type == "UTR"]
cds_sp  <- data.table(tx = as.character(cds_gr$transcript_id), s = start(cds_gr), e = end(cds_gr)
                      )[, .(cs = min(s), ce = max(e)), by = tx]
ut      <- data.table(i = seq_along(utr_gr), tx = as.character(utr_gr$transcript_id),
                      s = start(utr_gr), e = end(utr_gr), st = as.character(strand(utr_gr)))
ut      <- merge(ut, cds_sp, by = "tx", all.x = TRUE)

In [10]:
ut[, mid := (s + e) / 2]
ut[, side := fifelse(is.na(cs), NA_character_,
              fifelse(st == "+", fifelse(mid < cs, "5", "3"),
                                 fifelse(mid > ce, "5", "3")))]

In [11]:
u5      <- reduce(utr_gr[ut[side == "5", i]], ignore.strand = TRUE)
u3      <- reduce(utr_gr[ut[side == "3", i]], ignore.strand = TRUE)

In [12]:
genes   <- gtf[gtf$type == "gene"]
genes_r <- reduce(genes, ignore.strand = TRUE)
prom    <- reduce(suppressWarnings(promoters(tx, upstream = 3000, downstream = 3000)), ignore.strand = TRUE)
downs   <- reduce(suppressWarnings(flank(tx, width = 300, start = FALSE)), ignore.strand = TRUE)
introns <- GenomicRanges::setdiff(genes_r, exons)
tss     <- resize(tx, 1, fix = "start")            
rm(gtf); invisible(gc())

In [13]:
#  TE features + class_family (the RepeatMasker row-aligned with annotation)
ann   <- fread(file.path(DIR, "liu_repeat_counts-annotation.csv"))
rm_cf <- fread(file.path(DIR, "repeatmasker_raw.csv.gz"), select = c("rep_name","class_family"))
stopifnot(nrow(ann) == nrow(rm_cf))

In [14]:
ann[, `:=`(class_family = rm_cf$class_family, rep_name = rm_cf$rep_name)]

In [15]:
te <- GRanges(ann$Chr, IRanges(ann$Start, ann$End),
              strand = ifelse(ann$Strand %in% c("+","-"), ann$Strand, "*"))

# to assign a single, hierarchical genomic annotation category to each transposable element (TE) :

# genic-first region label: CDS > 5'UTR > 3'UTR > Exon > Intron > Promoter > Downstream > Intergenic 

In [16]:
# genic-first region label: CDS > 5'UTR > 3'UTR > Exon > Intron > Promoter > Downstream > Intergenic 

ov <- function(x, y) overlapsAny(x, y, ignore.strand = TRUE) ## Overlap testing
o_cds <- ov(te, cds_gr); o_u5 <- ov(te, u5); o_u3 <- ov(te, u3); o_ex <- ov(te, exons)
o_in  <- ov(te, introns); o_pr <- ov(te, prom); o_dn <- ov(te, downs)
ann[, region := fifelse(o_cds, "Exon",
             fifelse(o_u5, "5' UTR",
             fifelse(o_u3, "3' UTR",
             fifelse(o_ex, "Exon", # non-coding RNA exons or exons lacking explicit CDS/UTR labels in the annotation
             fifelse(o_in, "Intron",
             fifelse(o_pr, "Promoter (<=3kb)", # Regulatory region immediately upstream of the transcription start site (TSS)
             fifelse(o_dn, "Downstream", "Distal Intergenic")))))))] 

                        # Flanking region immediately downstream of the gene body / transcription termination site (TTS) (typically 1-3 kb).
                        # default fallback for any element that did not overlap any of the above gene-associated features

In [17]:
# mapping extras: overlap fractions, orientation, nearest gene + distance to TSS 
frac <- function(y) {
  bp <- numeric(length(te)); h <- findOverlaps(te, y, ignore.strand = TRUE)
  w  <- width(pintersect(te[queryHits(h)], y[subjectHits(h)], ignore.strand = TRUE))
  a  <- tapply(w, queryHits(h), sum); bp[as.integer(names(a))] <- as.numeric(a); bp / width(te)
}

In [18]:
ann[, `:=`(frac_exonic = round(frac(exons), 4), frac_intronic = round(frac(introns), 4))]
hg   <- findOverlaps(te, genes, ignore.strand = TRUE)
same <- as.character(strand(te))[queryHits(hg)] == as.character(strand(genes))[subjectHits(hg)]

In [19]:
sn <- logical(length(te)); an <- logical(length(te))
sn[unique(queryHits(hg)[same])] <- TRUE; an[unique(queryHits(hg)[!same])] <- TRUE

In [20]:
ann[, orientation := fifelse(sn & an, "both", fifelse(sn, "sense", fifelse(an, "antisense", "no_gene")))]
ng <- nearest(te, genes, ignore.strand = TRUE)

In [21]:
ann[, nearest_gene := ifelse(is.na(ng), NA_character_, genes$gene_name[ng])]
dt <- distanceToNearest(te, tss, ignore.strand = TRUE)

In [22]:
ann[, distanceToTSS := NA_integer_][queryHits(dt), distanceToTSS := mcols(dt)$distance]

In [23]:
# read in the counts, sample sheet, size factors here
ss   <- fread(file.path(DIR, "liu_MI_sample_sheet.csv"))
meta <- fread(file.path(DIR, "liu_sample_meta.csv"))
sf   <- setNames(meta$size_factor, meta$Run)

In [24]:
samples <- ss$srr; bamcols <- paste0(samples, ".sorted.bam")
counts  <- fread(file.path(DIR, "liu_repeat_counts-counts.csv"), select = c("GeneID", bamcols))
counts  <- merge(counts, ann[, .(GeneID, class_family, region)], by = "GeneID", sort = FALSE)
counts  <- counts[class_family %in% fams10]

In [25]:
# create down here the per-sample long table (raw + normalized)
agg  <- counts[, lapply(.SD, sum), by = .(class_family, region), .SDcols = bamcols]
long <- melt(agg, id.vars = c("class_family","region"), variable.name = "bam", value.name = "sum_raw_counts")
long[, SRR := sub("\\.sorted\\.bam$", "", bam)]
setnames(long, c("class_family","region"), c("TE_family","Genomic_location"))

In [26]:
full <- CJ(TE_family = fams10, Genomic_location = region_levels, SRR = samples, unique = TRUE)
long <- merge(full, long[, .(TE_family, Genomic_location, SRR, sum_raw_counts)],
              by = c("TE_family","Genomic_location","SRR"), all.x = TRUE)
long[is.na(sum_raw_counts), sum_raw_counts := 0]

In [27]:
smap <- ss[, .(SRR = srr, Sample = as.integer(sub("^S","",subject)),
               Group = fifelse(tdp43_status == "TDP43_pos","TDP43+","TDP43-"))]

In [28]:
long <- merge(long, smap, by = "SRR")
long[, size_factor := sf[SRR]]
long[, sum_normalized_counts := sum_raw_counts / size_factor]
long[, TE_family := factor(TE_family, levels = fams10)]
long[, Genomic_location := factor(Genomic_location, levels = region_levels)]

In [29]:
bysample <- long[, .(Group, Sample, SRR, TE_family, Genomic_location, size_factor,
                     sum_raw_counts = round(sum_raw_counts,2),
                     sum_normalized_counts = round(sum_normalized_counts,2))]

In [30]:
setorder(bysample, TE_family, Genomic_location, Sample, Group)
fwrite(bysample, "te_family_region_counts_by_sample_gencode47.csv")

In [31]:
# per-group summary
bygroup <- long[, .(sum_raw_counts = round(sum(sum_raw_counts),2),
                    mean_raw_per_sample = round(mean(sum_raw_counts),2),
                    sum_normalized_counts = round(sum(sum_normalized_counts),2),
                    mean_normalized_per_sample = round(mean(sum_normalized_counts),2)),
                 by = .(Group, TE_family, Genomic_location)]

In [32]:
setorder(bygroup, TE_family, Genomic_location, Group)
fwrite(bygroup, "te_family_region_counts_by_group_gencode47.csv")

In [33]:
# TE -> location mapping (10 families)
counts[, expr_total_raw := round(rowSums(.SD),3), .SDcols = bamcols]

In [34]:
mapping <- merge(ann[class_family %in% fams10,
                     .(GeneID, seqnames = Chr, start = Start, end = End, strand = Strand,
                       TE_family = class_family, rep_name, Length, Genomic_location = region,
                       frac_exonic, frac_intronic, orientation, nearest_gene, distanceToTSS)],
                 counts[, .(GeneID, expr_total_raw)], by = "GeneID", sort = FALSE)

In [35]:
fwrite(mapping, "te_location_mapping_10families_gencode47.csv.gz")

In [36]:
cat("by_sample:", nrow(bysample), "| by_group:", nrow(bygroup), "| mapping:", nrow(mapping), "\n")

by_sample: 980 | by_group: 140 | mapping: 3959317 
